In [0]:
from pyspark.sql.functions import (
    col,
    get_json_object,
    to_date,
    to_timestamp,
    regexp_replace,
    current_timestamp
)

CATALOG = "fhir_assignment"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:

#  Patient Silver production transformation

patient_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.patient")
    .select(
        col("resource_id").alias("patient_id"),

        get_json_object(
            col("resource_json"), "$.gender"
        ).alias("gender"),

        to_date(
            get_json_object(
                col("resource_json"), "$.birthDate"
            )
        ).alias("birth_date"),

        get_json_object(
            col("resource_json"), "$.name[0].family"
        ).alias("family_name"),

        get_json_object(
            col("resource_json"), "$.name[0].given[0]"
        ).alias("given_name"),

        get_json_object(
            col("resource_json"), "$.name[0].text"
        ).alias("name_text"),

        get_json_object(
            col("resource_json"), "$.identifier[0].value"
        ).alias("identifier_value"),

        get_json_object(
            col("resource_json"), "$.address[0].city"
        ).alias("city"),

        get_json_object(
            col("resource_json"), "$.address[0].state"
        ).alias("state"),

        get_json_object(
            col("resource_json"), "$.address[0].postalCode"
        ).alias("postal_code"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("record_hash"),
        col("run_id")
    )
)

In [0]:
#Encounter production transformation

encounter_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.encounter")
    .select(
        col("resource_id").alias("encounter_id"),

        get_json_object(
            col("resource_json"), "$.status"
        ).alias("status"),

        get_json_object(
            col("resource_json"), "$.class.code"
        ).alias("class_code"),

        get_json_object(
            col("resource_json"), "$.class.display"
        ).alias("class_display"),

        regexp_replace(
            get_json_object(
                col("resource_json"), "$.subject.reference"
            ),
            "^Patient/",
            ""
        ).alias("patient_id"),

        to_timestamp(
            get_json_object(
                col("resource_json"), "$.period.start"
            )
        ).alias("period_start"),

        to_timestamp(
            get_json_object(
                col("resource_json"), "$.period.end"
            )
        ).alias("period_end"),

        regexp_replace(
            get_json_object(
                col("resource_json"),
                "$.participant[0].individual.reference"
            ),
            "^Practitioner/",
            ""
        ).alias("practitioner_id"),

        get_json_object(
            col("resource_json"),
            "$.participant[0].individual.display"
        ).alias("practitioner_name"),

        get_json_object(
            col("resource_json"),
            "$.serviceProvider.display"
        ).alias("service_provider"),

        get_json_object(
            col("resource_json"),
            "$.serviceProvider.identifier.value"
        ).alias("facility_id"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("record_hash"),
        col("run_id")
    )
)

In [0]:
#  Observation production transformation

observation_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.observation")
    .select(
        col("resource_id").alias("observation_id"),

        get_json_object(
            col("resource_json"), "$.status"
        ).alias("status"),

        get_json_object(
            col("resource_json"),
            "$.category[0].coding[0].code"
        ).alias("category_code"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].code"
        ).alias("observation_code"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].display"
        ).alias("observation_display"),

        regexp_replace(
            get_json_object(
                col("resource_json"),
                "$.subject.reference"
            ),
            "^Patient/",
            ""
        ).alias("patient_id"),

        to_timestamp(
            get_json_object(
                col("resource_json"),
                "$.effectiveDateTime"
            )
        ).alias("effective_datetime"),

        get_json_object(
            col("resource_json"),
            "$.valueQuantity.value"
        ).cast("double").alias("value_quantity"),

        get_json_object(
            col("resource_json"),
            "$.valueQuantity.unit"
        ).alias("value_unit"),

        get_json_object(
            col("resource_json"),
            "$.valueString"
        ).alias("value_string"),

        get_json_object(
            col("resource_json"),
            "$.valueCodeableConcept.text"
        ).alias("value_codeable_text"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("record_hash"),
        col("run_id")
    )
)

In [0]:
# Condition production transformation

condition_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.condition")
    .select(
        col("resource_id").alias("condition_id"),

        get_json_object(
            col("resource_json"),
            "$.clinicalStatus.coding[0].code"
        ).alias("clinical_status"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].code"
        ).alias("condition_code"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].display"
        ).alias("condition_display"),

        get_json_object(
            col("resource_json"),
            "$.code.text"
        ).alias("condition_text"),

        regexp_replace(
            get_json_object(
                col("resource_json"),
                "$.subject.reference"
            ),
            "^Patient/",
            ""
        ).alias("patient_id"),

        regexp_replace(
            get_json_object(
                col("resource_json"),
                "$.encounter.reference"
            ),
            "^Encounter/",
            ""
        ).alias("encounter_id"),

        to_date(
            get_json_object(
                col("resource_json"),
                "$.onsetDateTime"
            )
        ).alias("onset_date"),

        col("source_file"),
        col("ingestion_timestamp"),
        col("record_hash"),
        col("run_id")
    )
)

patient_silver.printSchema()


 ONE-TIME SILVER TABLE INITIALIZATION


silver_tables = {
    "patient": patient_silver,
    "encounter": encounter_silver,
    "observation": observation_silver,
    "condition": condition_silver
}

for table_name, silver_df in silver_tables.items():

    full_table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Initialized Silver table: {full_table_name}")

 
 VERIFY SILVER TABLE SCHEMAS


SILVER_TABLES = {
    "patient": f"{CATALOG}.{SILVER_SCHEMA}.patient",
    "encounter": f"{CATALOG}.{SILVER_SCHEMA}.encounter",
    "observation": f"{CATALOG}.{SILVER_SCHEMA}.observation",
    "condition": f"{CATALOG}.{SILVER_SCHEMA}.condition"
}

for table_name, full_table_name in SILVER_TABLES.items():

    print(f"\n===== {full_table_name} =====")
    spark.table(full_table_name).printSchema()

In [0]:
# ============================================================
# WRITE SILVER TABLES - INCREMENTAL MERGE
# ============================================================

from delta.tables import DeltaTable

silver_tables = {
    "patient": patient_silver,
    "encounter": encounter_silver,
    "observation": observation_silver,
    "condition": condition_silver
}

silver_keys = {
    "patient": "patient_id",
    "encounter": "encounter_id",
    "observation": "observation_id",
    "condition": "condition_id"
}

for table_name, silver_df in silver_tables.items():

    full_table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    merge_key = silver_keys[table_name]

    target = DeltaTable.forName(
        spark,
        full_table_name
    )

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            f"target.{merge_key} = source.{merge_key}"
        )
        .whenMatchedUpdate(
            condition="target.record_hash <> source.record_hash",
            set={
                column: f"source.{column}"
                for column in silver_df.columns
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )